# 02a Byte BPE Generation

## Purpose
This notebook trains the byte-level BPE tokenizer for the rebuilt workflow.

## Why this notebook matters
This tokenizer serves as a simple baseline for the later tokenizer comparison work. It does not encode glycan-specific biological structure directly, so it provides a neutral reference point for the more structured tokenizers built later in the notebook family.

## Inputs
- The training split created by notebook 01

## Outputs
- Byte-BPE tokenizer files saved under `tokenizers/byte_bpe/<setting_label>/`
- `vocab.json`
- `merges.txt`
- `tokenizer_config_summary.json`
- optionally `inspection_preview.csv`


## Runtime setup

This cell prepares the Colab environment for the notebook. It mounts Google Drive, synchronizes the GitHub repository, and makes the project `src` code available for import.

We keep the setup code explicit here because the notebook must first download the repository before it can import the shared helper modules.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the GitHub repository is available locally
- confirmation of the active repository directory


In [ ]:
# Standard library imports used for environment setup.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read project data and save outputs.
drive.mount('/content/drive')

# Define the public GitHub repository that stores the project notebooks and
# shared helper modules.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = Path('/content') / REPO_NAME

# Clone the repository into the Colab runtime the first time the notebook runs.
# If the repository is already present, pull the latest changes so the notebook
# uses the current helper code.
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the Python import path so the notebook can import
# shared helper modules from the src package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository directory: {REPO_DIR}')


## User settings

This is the main cell that should be reviewed before running the notebook.

Update `PROJECT_ROOT` so it points to the correct project folder in Google Drive. This cell also exposes the tokenizer-specific settings that control the Byte-BPE training run and the saved output set.

**Settings to review**
- `PROJECT_ROOT`: the root folder for this project in Google Drive
- `TRAIN_SPLIT_FILENAME`: the text file used to train the tokenizer
- `OVERWRITE_EXISTING_OUTPUTS`: whether existing tokenizer artifacts may be replaced
- `VOCAB_SIZE`: target Byte-BPE vocabulary size
- `MIN_FREQUENCY`: minimum token count required for a merge candidate
- `SAVE_INSPECTION_PREVIEW`: whether to save the lightweight inspection CSV

**Expected output**
- the resolved training input path
- the tokenizer output directory
- the setting label used for this tokenizer run


In [ ]:
from pathlib import Path

from src.tokenizer_notebook_utils import build_tokenizer_output_paths, build_tokenizer_paths

# Update PROJECT_ROOT if your Drive project folder has a different name or
# location. This is the main path value that should be checked before running.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# This notebook trains on the training split created earlier in the workflow.
TRAIN_SPLIT_FILENAME = 'train.txt'

# If True, the notebook may replace previously saved tokenizer artifacts in the
# target output folder. If False, the notebook will stop before overwriting files.
OVERWRITE_EXISTING_OUTPUTS = False

# These settings control the Byte-BPE vocabulary size and the minimum token
# frequency required for merge learning.
VOCAB_SIZE = 300
MIN_FREQUENCY = 2

# Save a lightweight CSV preview of the tokenizer sanity check when True.
SAVE_INSPECTION_PREVIEW = True

SETTING_LABEL = f'v{VOCAB_SIZE}_m{MIN_FREQUENCY}'
TOKENIZER_FAMILY = 'byte_bpe'

# Build the standard paths used by this tokenizer notebook.
paths = build_tokenizer_paths(
    project_root=PROJECT_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    train_split_filename=TRAIN_SPLIT_FILENAME,
)
output_paths = build_tokenizer_output_paths(
    tokenizer_output_dir=paths['tokenizer_output_dir'],
    include_inspection_preview=SAVE_INSPECTION_PREVIEW,
    include_merges_file=True,
)

train_data_path = paths['train_data_path']
tokenizer_output_dir = paths['tokenizer_output_dir']
config_summary_path = paths['config_summary_path']
inspection_preview_path = paths['inspection_preview_path']

tokenizer_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Training data path: {train_data_path}')
print(f'Tokenizer output directory: {tokenizer_output_dir}')
print(f'Setting label: {SETTING_LABEL}')


## Validate the required paths

This cell checks that the training split exists and that the planned tokenizer output files can be written safely.

We do this early so that path problems and overwrite-policy problems are caught immediately instead of causing confusing downstream errors.

**Expected output**
- a confirmation that the training split exists
- a confirmation that the tokenizer output paths are valid for this run

**How to interpret the result**
- if this cell raises a file error, `PROJECT_ROOT` or `TRAIN_SPLIT_FILENAME` likely needs to be corrected
- if this cell raises a file-exists error, the notebook found prior tokenizer artifacts and overwrite mode is disabled


In [ ]:
from src.notebook_utils import require_existing_path, validate_output_paths

# Verify that the notebook can find the training split before continuing.
require_existing_path(train_data_path, 'Tokenizer training split')

# Enforce the shared overwrite policy before any tokenizer artifacts are saved.
validate_output_paths(
    output_paths=output_paths,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

print('Input and output path checks passed.')


## Train and save the Byte-BPE tokenizer

This cell trains the Byte-BPE tokenizer on the training split and saves both the low-level tokenizer files and the Hugging Face wrapper files.

**Expected output**
- a trained tokenizer saved to the tokenizer output directory
- a tokenizer summary JSON that records the main settings used for this run

**How to interpret the result**
- the tokenizer output directory should contain both the raw BPE files and the Hugging Face tokenizer files
- the saved summary JSON should reflect the same vocabulary and frequency settings shown in the notebook


In [ ]:
import json

from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

from src.tokenizer_notebook_utils import save_tokenizer_summary

print(f'Training byte-level BPE tokenizer (vocab={VOCAB_SIZE}, min_frequency={MIN_FREQUENCY})...')

# Train a Byte-BPE tokenizer directly on the glycan training strings.
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[str(train_data_path)],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=['<s>', '<pad>', '</s>', '<unk>', '<mask>'],
)

# Save the raw tokenizer model files such as vocab.json and merges.txt.
tokenizer.save_model(str(tokenizer_output_dir))

# Wrap the trained backend tokenizer in a Hugging Face fast tokenizer so later
# notebooks can load it through the standard Transformers interface.
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer._tokenizer,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>',
)

hf_tokenizer.save_pretrained(str(tokenizer_output_dir))

saved_files = sorted(path.name for path in tokenizer_output_dir.iterdir())
tokenizer_summary = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'setting_label': SETTING_LABEL,
    'vocab_size': VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'train_data_path': str(train_data_path),
    'tokenizer_output_dir': str(tokenizer_output_dir),
    'saved_files': saved_files,
}

save_tokenizer_summary(tokenizer_summary, config_summary_path)

print('Tokenizer training complete.')
print(f'Tokenizer saved to: {tokenizer_output_dir}')


## Load the saved tokenizer and inspect sample output

This cell performs a quick sanity check by loading the saved tokenizer and applying it to a few training examples.

We do not want deep analysis here. The goal is simply to confirm that the tokenizer loads successfully, has the expected special tokens, and produces token sequences that look plausible.

**Expected output**
- a small inspection table showing example sequences and tokenized output
- the loaded vocabulary size and key special tokens

**How to interpret the result**
- if token strings look badly fragmented or unexpectedly empty, the tokenizer training or save step may need review
- the loaded vocabulary size should be broadly consistent with the requested tokenizer settings


In [ ]:
from IPython.display import display
from transformers import PreTrainedTokenizerFast

from src.tokenizer_notebook_utils import (
    build_tokenizer_inspection_preview,
    load_training_sequences_for_tokenizer,
)

# Reload the saved tokenizer from disk so the sanity check uses the same files
# that later notebooks will load.
loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(str(tokenizer_output_dir))
train_sequences = load_training_sequences_for_tokenizer(train_data_path)
inspection_df = build_tokenizer_inspection_preview(
    tokenizer=loaded_tokenizer,
    sequences=train_sequences,
    num_samples=3,
    max_display_tokens=30,
)

display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')


## Save the inspection preview

This cell optionally saves a lightweight record of the tokenizer sanity check.

The saved preview is useful when you want a quick reminder of how the tokenizer behaved without reopening the notebook, but it is not required for downstream training.

**Expected output**
- when preview saving is enabled, a CSV file containing the inspection table
- when preview saving is disabled, a short confirmation message instead


In [ ]:
from src.tokenizer_notebook_utils import save_inspection_preview

if SAVE_INSPECTION_PREVIEW:
    saved_inspection_path = save_inspection_preview(
        inspection_df=inspection_df,
        output_path=inspection_preview_path,
    )
    print(f'Inspection preview saved to: {saved_inspection_path}')
else:
    print('Inspection preview saving is disabled for this run.')
